In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns, one_hot_encode
from src.pipelines import fit_and_predict, train_validate_models
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [2]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)

In [3]:
TRAFFIC_FEATURES_RAW = ["date", "sessions", "unique_visitors"]
SALE_FEATURES_RAW = ["date", "Revenue", "COGS"]

TRAFFIC_ENGINEER = ["date", "returning_rate"]

In [ ]:
df = pd.DataFrame(
   {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = df.merge(df_sales[SALE_FEATURES_RAW], on="date", how="left")
df = add_time_features(df, date_col="date")
# add feature year_after_shock = max(0, year - 2019)
# add feature shock_period = 1 if date is between 2019-03-01 and 2022-12-31, else 0

df["year_after_shock"] = (df["date"].dt.year - 2019).clip(lower=0)
df["shock_period"] = ((df["date"] >= "2019-03-01") & (df["date"] <= "2022-12-31")).astype(int)

# List all feature columns
feature_columns = list_feature_columns(df)
print(feature_columns["all_features"])
df.tail()

# Cần giải thích tại sao lag 30 cho revenue lại tệ
# 11 mạnh vượt trội, 12, 13, 14 cải thiện nhưng không lấy 
# 21, 28, 30, 10 làm tệ đi
df = add_lag_features(df, lags=[1,7,11], target_col="Revenue", date_col="date")




['day', 'month', 'year', 'day_of_week', 'week_of_year', 'month_sin', 'month_cos', 'is_month8_odd', 'Revenue_lag_1', 'Revenue_lag_7', 'Revenue_lag_11', 'year_after_shock', 'shock_period']


,date,Revenue,COGS,day,month,year,day_of_week,week_of_year,month_sin,month_cos,is_month8_odd,Revenue_lag_1,Revenue_lag_7,Revenue_lag_11,year_after_shock,shock_period
4195,2024-06-27,NaN,NaN,27,6,2024,3,26,1.224647e-16,-1.000000,False,NaN,NaN,NaN,5,0
4196,2024-06-28,NaN,NaN,28,6,2024,4,26,1.224647e-16,-1.000000,False,NaN,NaN,NaN,5,0
4197,2024-06-29,NaN,NaN,29,6,2024,5,26,1.224647e-16,-1.000000,False,NaN,NaN,NaN,5,0
4198,2024-06-30,NaN,NaN,30,6,2024,6,26,1.224647e-16,-1.000000,False,NaN,NaN,NaN,5,0
4199,2024-07-01,NaN,NaN,1,7,2024,0,27,-5.000000e-01,-0.866025,False,NaN,NaN,NaN,5,0


In [5]:

# categorical_cols = ["date_of_week", "month", "day", "is_weekend",
#                    "shock_period"]
# df = one_hot_encode(df, categorical_cols)

In [9]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

result = train_validate_models(
    df=df,
    model_config=model_config,
    train_range=("2013-01-01", "2021-12-31"),
    predict_range=("2022-01-01", "2022-12-31"),
)

result["metrics"]



{'Revenue': {'mae': 534243.2045718126,
  'rmse': 735123.8242972465,
  'mape': 19.02303376292975,
  'smape': 18.28535341087737,
  'r2': 0.8071104385356929},
 'COGS': {'mae': 481947.8034379652,
  'rmse': 657629.7920809386,
  'mape': 19.32976759747756,
  'smape': 18.788789603362005,
  'r2': 0.796712832986018}}

In [10]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

submission = fit_and_predict(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)
submission.head()
submission_name = "miracle"
submission.to_csv(project_root / f"submissions/{submission_name}.csv", index=False)